In [ ]:
# prep the PhosphoSitePlus API stuff and start to assess what Uniprot IDs don't match between the two sets. 

# prep and create the same structure for PTMs taken from UniProt. 

# THen, go through and add PTMs from the two resources. To do this, will assess whetehr it's a new PTM or an existing one. (have to create a map between PTM types in UniProt and PhosphoSitePlus).
# if it's a new PTM, insert it and the evidence. If it's an existing PTM, add the evidence to the existing PTM.

import PhosphoSitePlus_Tools

PSITE_DIR = '/Users/kmn4mj/GIT/Data/PhosphoSitePlus/2025_Dec31/'

PhosphoSitePlus_Tools.convert_pSiteDataFiles(PSITE_DIR, PSITE_DIR + 'phosphositeplus_data_fixed.csv')



In [3]:
# prepare now for integration of PhosphSitePlus data into ProteomeScout data file.
 
#let's just compare human to human for now.
import pandas as pd

pscout_data_dir = '/Users/kmn4mj/GIT/Data/proteomescout_everything_20190701/proteomescout_everything_20251229/'
pscout_data_file = pscout_data_dir + 'data_human_updated.tsv'

pscout = pd.read_csv(pscout_data_file, sep='\t')

PSITE_DIR = '/Users/kmn4mj/GIT/Data/PhosphoSitePlus/2025_Dec31/'
psite_file = PSITE_DIR + 'phosphositeplus_data.csv'
psite = pd.read_csv(psite_file)

In [ ]:
# let's just look at human proteins for now.
psite_human = psite[psite['species'] == 'human']



In [ ]:
uniprot_ids_pscout = set(pscout['uniprot_id'].unique())
uniprot_ids_psite = set(psite_human['Uniprot_ID'].unique())
uniprot_ids_both = uniprot_ids_pscout.intersection(uniprot_ids_psite)
print("ProteomeScout Uniprot IDs: %d"%(len(uniprot_ids_pscout)))
print("PhosphoSitePlus Uniprot IDs: %d"%(len(uniprot_ids_psite)))
print("Uniprot IDs in both: %d"%(len(uniprot_ids_both)))
# OK, so I have to handle adding proteins that are not yet in ProteomeScout, along with updating sites for those that are..

ProteomeScout Uniprot IDs: 22742
PhosphoSitePlus Uniprot IDs: 26761
Uniprot IDs in both: 21009


In [9]:
# what are the species in psite?
for species in psite['species'].unique():
    print("Species: %s, Count: %d"%(species, len(psite[psite['species']==species])))


Species: mouse, Count: 20605
Species: human, Count: 26770
Species: rat, Count: 14775
Species: adenovirus, Count: 2
Species: cow, Count: 578
Species: frog, Count: 9
Species: pig, Count: 108
Species: dog, Count: 42
Species: rabbit, Count: 95
Species: sheep, Count: 16
Species: chicken, Count: 96
Species: hamster, Count: 18
Species: turkey, Count: 2
Species: bacteria, Count: 40
Species: starfish, Count: 1
Species: horse, Count: 5
Species: SARSCoV1, Count: 31
Species: guinea pig, Count: 3
Species: permutotetravirus, Count: 1
Species: herpesvirus, Count: 10
Species: SARSCoV2, Count: 32
Species: quail, Count: 2
Species: marmoset, Count: 1
Species: goat, Count: 4
Species: green monkey, Count: 1
Species: mimivirus, Count: 1
Species: fruit fly, Count: 19
Species: torpedo, Count: 2
Species: polyomavirus, Count: 7
Species: MERSCoV, Count: 6
Species: caudovirus, Count: 3
Species: vaccinia virus, Count: 3
Species: ferret, Count: 1
Species: firefly, Count: 1
Species: cat, Count: 2
Species: retrovirus

In [17]:
# check to see if we have redundant sequences in the pscout data file
df = psite_human
uniprot_name = 'Uniprot_ID'
unique_sequences = df['sequence'].unique()
print("ProteomeScout unique sequences: %d, total entries: %d"%(len(unique_sequences), len(df)))
# OK, let's look at the redundant sequences
sequence_counts = df['sequence'].value_counts()
redundant_sequences = sequence_counts[sequence_counts > 1]
print("Redundant sequences count: %d"%(len(redundant_sequences)))
for name, group in df.groupby('sequence'):
    if len(group) > 1:
        #print("Sequence: %s, Count: %d"%(name, len(group)))
        print(group[[uniprot_name, 'modifications']])

ProteomeScout unique sequences: 26667, total entries: 26770
Redundant sequences count: 70
      Uniprot_ID                                      modifications
17953     P86791  S10-Phosphoserine;S76-Phosphoserine;S136-Phosp...
52658     P86790  S10-Phosphoserine;S76-Phosphoserine;S136-Phosp...
      Uniprot_ID                                      modifications
16698     Q6P6C2  S5-Phosphoserine;Y51-Phosphotyrosine;S54-Phosp...
28607   Q6P6C2-2  S371-Phosphoserine;S374-Phosphoserine;S384-Pho...
      Uniprot_ID modifications
19416     P0DMW5           NaN
19418     P0DMW4           NaN
      Uniprot_ID                                      modifications
24969     C9J9J2  S418-Phosphoserine;S450-Phosphoserine;R345-Met...
39513     Q6ZMS7  T101-Phosphothreonine;Y106-Phosphotyrosine;Y16...
      Uniprot_ID                                      modifications
28989     Q9BV68  S5-Phosphoserine;S17-Phosphoserine;S293-Phosph...
52366   Q9BV68-2  T278-Phosphothreonine;T286-Phosphothreonine;S2...
 